In [ ]:
%pip install eyepop

In [ ]:
import getpass

EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')


### Define the Pop
This pipeline reads license plates without a VLM ability, by chaining pretrained object detection and OCR abilities:

1. `eyepop.vehicle:latest` detects vehicles.
2. Each vehicle crop is forwarded to `eyepop.vehicle.license-plate:latest`, which detects the plate (a `tracking` branch runs alongside it so vehicles keep a stable identity across video frames).
3. Each plate crop is forwarded to `eyepop.text.recognize.landscape:latest`, an OCR ability that reads the plate text.

No custom ability needs to be trained or published for this to work — every component below is a public, pretrained ability.

In [ ]:
from eyepop import EyePopSdk
from eyepop.worker.worker_types import (
    CropForward,
    InferenceComponent,
    InferenceType,
    MotionModel,
    Pop,
    TrackingComponent,
)
import json

pop = Pop(components=[
    InferenceComponent(
        ability="eyepop.vehicle:latest",
        inferenceTypes=[InferenceType.OBJECT_DETECTION],
        categoryName="Vehicle",
        confidenceThreshold=0.8,
        forward=CropForward(
            targets=[
                TrackingComponent(
                    maxAgeSeconds=6,
                    agnostic=True,
                    motionModel=MotionModel.CONSTANT_VELOCITY,
                    classHysteresis=True,
                ),
                InferenceComponent(
                    ability="eyepop.vehicle.license-plate:latest",
                    inferenceTypes=[InferenceType.OBJECT_DETECTION],
                    categoryName="License Plate",
                    confidenceThreshold=0.2,
                    forward=CropForward(
                        targets=[
                            InferenceComponent(
                                ability="eyepop.text.recognize.landscape:latest",
                                inferenceTypes=[InferenceType.OCR],
                            )
                        ]
                    ),
                ),
            ]
        ),
    )
])

### Evalulate on a Single Image

In [ ]:
from pathlib import Path

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY, pop=pop) as endpoint:
   sample_img_path = Path("./image/example.webp")
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")